In [6]:
# Install packages
# %pip install requests
# %pip install bs4
# %pip install googlesearch-python


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py) ... done
  Created wheel for googlesearch-python: filename=googlesearch_python-1.2.3-py3-none-any.whl size=4229 sha256=aabf0357a49a6a798fd985e3327f817a844deef28704a6f70017547b4043a8ab
  Stored in directory: /Users/modekurt/Library/Caches/pip/wheels/be/78/a3/d80e85ac9551489cd9c44f59a0bc5972e79e680ce9cc6055ca
Successfully built googlesearch-python
Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
from bs4 import BeautifulSoup
from googlesearch import search
import os
import pandas as pd

In [5]:
# Extract Municipal Website from City Wikipedia Page
# Example: Sacramento, CA

def get_sacramento_municipal_website_link():
    # Wikipedia URL for Sacramento
    wikipedia_url = 'https://en.wikipedia.org/wiki/Sacramento,_California'

    # Send a request to the Wikipedia page
    response = requests.get(wikipedia_url)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content of the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the right panel section
        right_panel = soup.find('table', {'class': 'infobox'})

        # Check if the right panel section exists
        if right_panel:
            # Find the row containing the 'Website' label
            website_row = right_panel.find('th', string='Website')

            # Check if the row is found
            if website_row:
                # Get the link from the corresponding 'td' element
                link = website_row.find_next('td').find('a', href=True)

                # Check if a link is found
                if link:
                    return link['href']
                else:
                    return "Municipal website link not found in the right panel."
            else:
                return "'Website' label not found in the right panel."
        else:
            return "Right panel section not found on the Wikipedia page."
    else:
        return f"Error accessing the Wikipedia page. Status code: {response.status_code}"

# Example usage for Sacramento
sacramento_municipal_website_link = get_sacramento_municipal_website_link()

print(f"Municipal Website Link for Sacramento: {sacramento_municipal_website_link}")


Municipal Website Link for Sacramento: http://cityofsacramento.org


In [16]:
#Now search on google for climate action plans using extracted link

def download_search_results(query, city_code, num_results):
    # Perform Google search and get the first 'num_results' URLs
    #I think th
    search_results = list(search(query, num_results=num_results))
    print(search_results)

    # Create a directory to save the PDFs
    if not os.path.exists('downloaded_pdfs'):
        os.makedirs('downloaded_pdfs')

    # Download content from each URL
    for i, url in enumerate(search_results):
        try:
            response = requests.get(url)
            print (response)
            # Save the content to a file (you can modify this part based on your needs)
            with open(f'test.pdf', 'wb') as pdf_file:
                pdf_file.write(response.content)
            print(f"Downloaded result {i + 1}: {url}")
        except Exception as e:
            print(f"Error downloading result {i + 1}: {url}\nError: {e}")

# Example usage
search_query = sacramento_municipal_website_link + 'climate adaptation plan' + 'filetype:pdf'
download_search_results(search_query, num_results=1)

['https://www.cityofsacramento.org/Community-Development/Planning/Major-Projects/General-Plan/About-The-Project/Climate_Change', 'https://www.cityofsacramento.org/-/media/Corporate/Files/CMO/Climate-Change/2021ClimateImplementationPlan_DRAFT.pdf?la=en']
<Response [200]>
Downloaded result 1: https://www.cityofsacramento.org/Community-Development/Planning/Major-Projects/General-Plan/About-The-Project/Climate_Change
<Response [200]>
Downloaded result 2: https://www.cityofsacramento.org/-/media/Corporate/Files/CMO/Climate-Change/2021ClimateImplementationPlan_DRAFT.pdf?la=en


In [ ]:
#TODO: get and check translations 
#TODO: look to literature for other search queries
search_queries = {'english': ["climate adaptation plan", "climate action plan"], 'spanish': ["Plan de Acción Climática"] } 
city_list = pd.read_csv('output/capital_and_1mill_pop_cities.csv')

def plan_search(row):
    query = search_queries[row['language'].tolower()] + row['City']
    city_code = row['City Code']
    download_search_results(query, city_code, 10)
    return 

#look for searches for each city
searched_plans = city_list.apply(plan_search)